# Ch 27. Flash Attention — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: PyTorch SDPAのバックエンドを`math`、`mem_efficient`に明示設定して比較せよ。

### 解法

同一の Q/K/V に対して `math` と `mem_efficient` を個別に強制する。現在の device または PyTorch build が backend を未対応なら例外を結果に明示し、実行できた backend は math に対する最大絶対誤差を計算する。これにより CPU 上で math へ暗黙に fallback することを防ぐ。

以下の code は download を行わず、実際の backend 可否と数値誤差を出力する。


In [ ]:
import torch
from torch.nn.functional import scaled_dot_product_attention

torch.manual_seed(27)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
q = torch.randn(1, 2, 32, 16, device=device)

try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    requested = {'math': SDPBackend.MATH, 'mem_efficient': SDPBackend.EFFICIENT_ATTENTION}
    def forced(backend):
        with sdpa_kernel(backend):
            return scaled_dot_product_attention(q, q, q)
except ImportError:  # PyTorch 2.0 compatibility
    from torch.backends.cuda import sdp_kernel
    requested = {'math': 'math', 'mem_efficient': 'mem_efficient'}
    def forced(backend):
        flags = {'enable_math': backend == 'math',
                 'enable_mem_efficient': backend == 'mem_efficient',
                 'enable_flash': False}
        with sdp_kernel(**flags):
            return scaled_dot_product_attention(q, q, q)

outputs, comparison = {}, {'device': device}
for name, backend in requested.items():
    try:
        outputs[name] = forced(backend)
        comparison[name] = {'status': 'executed', 'shape': tuple(outputs[name].shape)}
    except RuntimeError as error:
        comparison[name] = {'status': 'unsupported', 'reason': str(error).splitlines()[0]}

assert comparison['math']['status'] == 'executed'
if 'mem_efficient' in outputs:
    max_error = float((outputs['math'] - outputs['mem_efficient']).abs().max())
    comparison['mem_efficient']['max_abs_error_vs_math'] = max_error
    assert torch.allclose(outputs['math'], outputs['mem_efficient'], atol=2e-5, rtol=2e-5)
comparison


## 問題 2

**問題**: 系列長1024、4096、16384で標準attentionとSDPAの時間・メモリを比較せよ。

### 解法

標準attentionは$n\times n$ scoreを保存しmemoryは$O(n^2)$。tile型SDPAはmaterializeを避け小block workspaceを使うが、実時間・memoryはOOMも結果としてdevice別に測る。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
lengths=[1024,4096,16384]; standard_scores=[n*n for n in lengths]; tiled_scores=[n*128 for n in lengths]
assert standard_scores[-1]/tiled_scores[-1]==128
list(zip(lengths,standard_scores,tiled_scores))


## 問題 3

**問題**: Online Softmaxを実装し、標準softmaxと結果が一致することを示せ。

### 解法

新しい$x$ごとに$m'=\max(m,x)$、$l'=l e^{m-m'}+e^{x-m'}$と更新しoverflowなしで分母を得る。確率和と要素誤差を検証する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
x=np.array([1000.,999.,998.,997.]); m=-np.inf; denom=0.
for v in x:
 new=max(m,v); denom=denom*np.exp(m-new)+np.exp(v-new); m=new
p=np.exp(x-m)/denom; ref=np.exp(x-x.max()); ref/=ref.sum()
assert np.allclose(p,ref); p


## 問題 4

**問題**: Flash Attentionが逆伝播メモリをどのように節約するか説明せよ。

### 解法

Flash Attentionは順伝播の全score/確率行列を保存せずblockごとに再計算する。追加計算によりactivation保存を$O(n^2)$から概ね$O(nd)$へ下げるtradeoffである。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
def memory_elements(n,d,block): return {'standard_scores':n*n,'flash_workspace':block*block+n*d}
r=memory_elements(4096,64,128); assert r['flash_workspace']<r['standard_scores']; r


## 問題 5

**問題**: Ring Attentionが複数GPUでどのように動作するか説明せよ。

### 解法

各deviceはlocal Qを保持しK/V blockをringで$N-1$回送る。各段階のonline-softmax統計をmergeすると全K/Vに対する正確なattentionを得て、device当たり保存は約$1/N$となる。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
devices=4; blocks=list(range(devices)); seen=[]
for step in range(devices): seen.append([blocks[(rank-step)%devices] for rank in range(devices)])
assert all(set(seen[s])==set(range(devices)) for s in range(devices)); seen
